### This is an experiment for new image-text retrieval web ###
NOTES:
- This experiment uses Apple's version of CLIP, namely "DFN5B CLIP ViT H14"
- The following part is only for inference computations. To experiment feature extracting, go to feat-extractor.ipynb

**SETUP**

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from open_clip import create_model_from_pretrained, get_tokenizer
from pymilvus import MilvusClient

model, preprocess = create_model_from_pretrained('hf-hub:apple/DFN5B-CLIP-ViT-H-14-384')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# model.eval()  later
model.to(device)
tokenizer = get_tokenizer('ViT-H-14')

c:\Users\uwu\AppData\Local\Programs\Python\Python312\Lib\site-packages\open_clip\factory.py:129: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkp

**CONNECT TO CLIENT**

In [2]:
client = MilvusClient(
    uri='https://in03-cbdb10d1d199984.serverless.aws-eu-central-1.cloud.zilliz.com',
    token='6305ff67695e59bf211ca717cb68f74f7367ccfd56195824ba6aad7c07f5924cc6c9da8aadec4354aedc193a997225494903a9d7'
)

**TEXT QUERY PREPARATION**

In [ ]:
text = 'A display of a lot of golden shoes arranged in order.'
text = tokenizer(text, context_length=model.context_length)
with torch.no_grad(), torch.amp.autocast('cuda'):
    text_feature = model.encode_text(text)
    text_feature = F.normalize(text_feature, dim=-1)

text_feature = text_feature.squeeze().tolist()

c:\Users\uwu\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\amp\autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


In [4]:
print(text_feature)

[-0.01857038028538227, 0.019362321123480797, -0.0022334442473948, 0.03300067037343979, 0.013445127755403519, 0.0436200313270092, -0.005155069287866354, 0.008670171722769737, 0.015434092842042446, -0.014424482360482216, -0.0787288099527359, -0.009409517981112003, -0.007082074414938688, -0.012031442485749722, 0.07365681976079941, -0.029972510412335396, 0.04813709482550621, 0.08481927961111069, 0.016612187027931213, 0.011881952174007893, 0.028057733550667763, 0.0020903891418129206, -0.042681705206632614, 0.01957760751247406, -0.01205444522202015, -0.04054823890328407, 0.1815182864665985, 0.06633877754211426, -0.007578888908028603, 0.025767061859369278, -0.012762670405209064, -0.0283666979521513, -0.04311439022421837, -0.04661315307021141, 0.026831049472093582, 0.007637422997504473, 0.02335520274937153, -0.02955208346247673, 0.021441645920276642, -0.040916189551353455, -0.0040685259737074375, -0.003463227301836014, 0.043893180787563324, -0.01934286393225193, -0.008852523751556873, -0.00553

**SEARCH PARAMS AND TOP K PREPARATION**

In [6]:
TOP_K = 100
search_params = {
    'metric_type': 'COSINE',
    'params': {
        'ef': 200
    }
}

**IMAGE SEARCH**

In [10]:
results = client.search(
    "aictestbatch",
    data=[text_feature],
    limit=TOP_K,
    output_fields=["img_key"],
    search_params=search_params
)

**SHOW QUERY RESULT**

In [43]:
for res in results:
    for hit in res:
        original_path = f'D:\\AIC2024\\kfs\\{hit['entity']['img_key'].replace('/', '\\')[5:]}'
        print(original_path)

D:\AIC2024\kfs\\Keyframes_L22\keyframes\L22_V011\231.jpg
D:\AIC2024\kfs\\Keyframes_L29\keyframes\L29_V002\373.jpg
D:\AIC2024\kfs\\Keyframes_L09\keyframes\L09_V023\285.jpg
D:\AIC2024\kfs\\Keyframes_L02\keyframes\L02_V003\184.jpg
D:\AIC2024\kfs\\Keyframes_L13\keyframes\L13_V021\199.jpg
D:\AIC2024\kfs\\Keyframes_L14\keyframes\L14_V019\213.jpg
D:\AIC2024\kfs\\Keyframes_L18\keyframes\L18_V014\314.jpg
D:\AIC2024\kfs\\Keyframes_L16\keyframes\L16_V009\198.jpg
D:\AIC2024\kfs\\Keyframes_L24\keyframes\L24_V026\083.jpg
D:\AIC2024\kfs\\Keyframes_L12\keyframes\L12_V027\169.jpg
D:\AIC2024\kfs\\Keyframes_L19\keyframes\L19_V018\198.jpg
D:\AIC2024\kfs\\Keyframes_L18\keyframes\L18_V014\313.jpg
D:\AIC2024\kfs\\Keyframes_L24\keyframes\L24_V006\095.jpg
D:\AIC2024\kfs\\Keyframes_L13\keyframes\L13_V011\383.jpg
D:\AIC2024\kfs\\Keyframes_L02\keyframes\L02_V003\185.jpg
D:\AIC2024\kfs\\Keyframes_L06\keyframes\L06_V020\069.jpg
D:\AIC2024\kfs\\Keyframes_L10\keyframes\L10_V020\202.jpg
D:\AIC2024\kfs\\Keyframes_L24\k

In [19]:
for res in results: print(res)

[{'id': 459941207430762992, 'distance': 0.3183068037033081, 'entity': {'img_key': 'aic24/Keyframes_L22/keyframes/L22_V011/231.jpg'}}, {'id': 459941207431044884, 'distance': 0.3154304623603821, 'entity': {'img_key': 'aic24/Keyframes_L29/keyframes/L29_V002/373.jpg'}}, {'id': 459941207426165574, 'distance': 0.30582141876220703, 'entity': {'img_key': 'aic24/Keyframes_L09/keyframes/L09_V023/285.jpg'}}, {'id': 459941207426125692, 'distance': 0.3056768774986267, 'entity': {'img_key': 'aic24/Keyframes_L02/keyframes/L02_V003/184.jpg'}}, {'id': 459941207430717090, 'distance': 0.30093538761138916, 'entity': {'img_key': 'aic24/Keyframes_L13/keyframes/L13_V021/199.jpg'}}, {'id': 459941207430721411, 'distance': 0.2902981638908386, 'entity': {'img_key': 'aic24/Keyframes_L14/keyframes/L14_V019/213.jpg'}}, {'id': 459941207430938885, 'distance': 0.28188687562942505, 'entity': {'img_key': 'aic24/Keyframes_L18/keyframes/L18_V014/314.jpg'}}, {'id': 459941207426397431, 'distance': 0.2814439535140991, 'entit

**DEBUGGING**

In [33]:
print(image_features)

[tensor([[ 0.0147,  0.0590,  0.0068,  ..., -0.0162,  0.0381, -0.0464]]), tensor([[-0.0498,  0.0538,  0.0024,  ..., -0.0378,  0.0126,  0.0075]]), tensor([[-0.0303,  0.0669,  0.0074,  ..., -0.0443,  0.0313,  0.0123]]), tensor([[-0.0127,  0.0011, -0.0166,  ..., -0.0186, -0.0162, -0.0293]]), tensor([[-0.0110, -0.0096, -0.0168,  ..., -0.0192, -0.0184, -0.0086]]), tensor([[ 0.0251, -0.0248,  0.0363,  ...,  0.0042,  0.0288, -0.0665]]), tensor([[-0.0083,  0.0057, -0.0030,  ..., -0.0009,  0.0565, -0.0442]]), tensor([[ 0.0149,  0.0714,  0.0181,  ..., -0.0263, -0.0030, -0.0037]]), tensor([[-0.0280,  0.0663,  0.0092,  ..., -0.0364,  0.0161,  0.0038]]), tensor([[-0.0221,  0.0467, -0.0003,  ..., -0.0395,  0.0256, -0.0271]]), tensor([[-0.0052,  0.0448,  0.0105,  ..., -0.0446,  0.0263, -0.0287]]), tensor([[-0.0098,  0.0169, -0.0102,  ..., -0.0153,  0.0190, -0.0379]]), tensor([[ 0.0034,  0.0204, -0.0070,  ..., -0.0229,  0.0140, -0.0246]]), tensor([[ 0.0073, -0.0102,  0.0052,  ..., -0.0114,  0.0284, -0.